# Vietnamese → English Translation Fine-Tuning (Colab)

This notebook fine-tunes `Qwen/Qwen3-1.7B` on the `viamr` Vietnamese→English data (AMR + sentence → English) and saves everything to Google Drive.

**Prerequisites**
- Upload the whole project folder (the one that contains `viamr/`, `data/`, `scripts/`, `requirements.txt`) to Google Drive.
- Runtime → Change runtime type → **GPU** (A100/L4/T4).

The notebook works on a single GPU; DeepSpeed is disabled and `torchrun` is not used.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

Set `DRIVE_PROJECT_DIR` to the folder in your Drive that holds this repo (the folder that contains `viamr/`, `data/`, `requirements.txt`). All checkpoints, logs, and results are stored under `DRIVE_OUTPUT_DIR` in Drive so they survive runtime restarts.

In [ ]:
import os

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/translate'
DRIVE_OUTPUT_DIR  = '/content/drive/MyDrive/translate_outputs'

WORK_DIR = '/content/translate'

MODEL_NAME     = 'Qwen/Qwen3-1.7B'
TRAIN_JSONL    = 'data/train.jsonl'
VALID_JSONL    = 'data/tst2012.jsonl'
TEST_JSONL     = 'data/tst2013.jsonl'

SFT_OUTPUT_DIR   = os.path.join(DRIVE_OUTPUT_DIR, 'Qwen-1.7B-SFT-VI2EN')
GRPO_OUTPUT_DIR  = os.path.join(DRIVE_OUTPUT_DIR, 'Qwen-1.7B-GRPO-VI2EN')
RESULTS_FILE     = os.path.join(DRIVE_OUTPUT_DIR, 'results.jsonl')

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
assert os.path.isdir(DRIVE_PROJECT_DIR), f'Project dir not found in Drive: {DRIVE_PROJECT_DIR}'
print('Project:', DRIVE_PROJECT_DIR)
print('Output (Drive):', DRIVE_OUTPUT_DIR)

## 3. Copy project into local workspace

Training from a Drive-mounted path is slow and fragile. We copy the code+data to local disk (`/content/translate`) and work there. Only outputs are written back to Drive.

In [ ]:
import shutil

if os.path.isdir(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(DRIVE_PROJECT_DIR, WORK_DIR)
os.chdir(WORK_DIR)
print('cwd:', os.getcwd())
!ls

## 4. Install dependencies

Colab already ships recent `torch` and `transformers`; we add `trl`, `peft`, `sacrebleu`, `datasets`, and `accelerate`. `deepspeed`, `vllm`, and `mpi4py` from `requirements.txt` are skipped — they are not needed for single-GPU Colab runs.

In [ ]:
!pip install -q -U transformers==4.46.3 trl==0.12.1 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0 sacrebleu==2.4.3

## 5. Sanity-check GPU and data

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
!head -1 data/train.jsonl
!wc -l data/train.jsonl data/tst2012.jsonl data/tst2013.jsonl

## 6. Preview the prompt that the model will see

This double-checks the dataset loader is picking `input` (AMR), `vi`, and `output` correctly.

In [ ]:
from viamr.dataset import get_data

ds = get_data(TRAIN_JSONL, type='sft')
print('Examples:', len(ds))
ex = ds[0]
print('--- system ---')
print(ex['prompt'][0]['content'])
print('--- user ---')
print(ex['prompt'][1]['content'])
print('--- target ---')
print(ex['completion'][0]['content'])

## 7. SFT fine-tuning

We call the training entrypoint as a Python module so the same code path runs here and in `scripts/train_sft.sh`. Tune `PER_DEVICE_BS`, `GRAD_ACC`, and `EPOCHS` to fit your GPU.

In [ ]:
import os, sys, subprocess

PER_DEVICE_BS = 2
GRAD_ACC      = 8
EPOCHS        = 3
LR            = 2e-5
MAX_LEN       = 1024
USE_LORA      = 1  # 1 = LoRA (fast, low VRAM); 0 = full fine-tune

cmd = [
    sys.executable, '-m', 'viamr.training.sft',
    '--dataset1_path', TRAIN_JSONL,
    '--output_dir',    SFT_OUTPUT_DIR,
    '--model_name',    MODEL_NAME,
    '--learning_rate', str(LR),
    '--warmup_steps',  '200',
    '--lr_scheduler_type', 'cosine',
    '--logging_steps', '10',
    '--per_device_train_batch_size', str(PER_DEVICE_BS),
    '--gradient_accumulation_steps', str(GRAD_ACC),
    '--max_input_length', str(MAX_LEN),
    '--num_train_epochs', str(EPOCHS),
    '--save_steps', '500',
    '--use_lora', str(USE_LORA),
    '--lora_r', '32', '--lora_alpha', '64', '--lora_dropout', '0.1',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 8. Inference on the test set

Runs the trained checkpoint on `data/tst2013.jsonl` and writes `results.jsonl` to Drive.

In [ ]:
import sys, subprocess

cmd = [
    sys.executable, '-m', 'viamr.inference',
    '--input_file',  TEST_JSONL,
    '--output_file', RESULTS_FILE,
    '--model_name',  SFT_OUTPUT_DIR,
    '--max_new_tokens', '512',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 9. Corpus BLEU

In [ ]:
import sys, subprocess

subprocess.run([
    sys.executable, '-m', 'viamr.scoring',
    '--predict_file', RESULTS_FILE,
], check=True)

## 10. (Optional) GRPO stage on top of SFT

GRPO uses sentence BLEU as the reward. It starts from the SFT checkpoint and is slower — expect ~4× the SFT wall-clock per step because each prompt is sampled `num_generations` times.

In [ ]:
import sys, subprocess

cmd = [
    sys.executable, '-m', 'viamr.training.grpo',
    '--dataset1_path', TRAIN_JSONL,
    '--output_dir',    GRPO_OUTPUT_DIR,
    '--model_name',    SFT_OUTPUT_DIR,
    '--learning_rate', '1e-6',
    '--warmup_steps',  '100',
    '--lr_scheduler_type', 'cosine',
    '--logging_steps', '1',
    '--per_device_train_batch_size', '2',
    '--gradient_accumulation_steps', '8',
    '--num_generations', '4',
    '--max_prompt_length',     '512',
    '--max_completion_length', '256',
    '--num_train_epochs', '1',
    '--save_steps', '200',
    '--use_lora', '1',
    '--lora_r', '16', '--lora_alpha', '32', '--lora_dropout', '0.1',
    '--wandb_project', 'vi2en-translation',
    '--wandb_run_name', 'grpo-bleu-colab',
]
os.environ.setdefault('WANDB_MODE', 'disabled')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 11. Where are my files?

- SFT checkpoint: `DRIVE_OUTPUT_DIR/Qwen-1.7B-SFT-VI2EN`
- GRPO checkpoint: `DRIVE_OUTPUT_DIR/Qwen-1.7B-GRPO-VI2EN`
- Translations: `DRIVE_OUTPUT_DIR/results.jsonl`

Everything lives in Drive, so you can close the runtime without losing work.